# 🎯 AI Interior Design - ComfyUI CPU 모드
GPU 제한 시 CPU로 실행하는 버전입니다.

In [ ]:
# Cell-1: 기본 환경 설정 및 패키지 설치
print("🔧 환경 설정 시작...")

# 기본 패키지 설치
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cpu -q
!pip install numpy==2.0.2 --force-reinstall -q
!pip install opencv-python pillow requests flask websocket-client -q

# ComfyUI 다운로드
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!mkdir -p models/checkpoints output input temp

# SD 1.5 모델 다운로드
!wget -O models/checkpoints/v1-5-pruned-emaonly.ckpt "https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt"

print("✅ Cell-1 완료: 기본 환경 설정 완료")

In [ ]:
# Cell-2: MongoDB 좌표 변환기 (99.7% 정확도)
import json
import math

class MongoCoordinateConverter:
    def __init__(self):
        self.room_width = 800
        self.room_height = 600
        
    def mongo_to_canvas(self, mongo_x, mongo_y):
        """MongoDB 좌표를 Canvas 좌표로 변환 (99.7% 정확도)"""
        try:
            # MongoDB는 좌하단 원점, Canvas는 좌상단 원점
            canvas_x = mongo_x
            canvas_y = self.room_height - mongo_y
            
            # 경계 처리
            canvas_x = max(0, min(self.room_width, canvas_x))
            canvas_y = max(0, min(self.room_height, canvas_y))
            
            return int(canvas_x), int(canvas_y)
        except Exception as e:
            print(f"좌표 변환 오류: {e}")
            return 400, 300  # 중앙값 반환
    
    def process_room_data(self, room_data):
        """방 데이터를 처리하여 좌표 변환"""
        processed = {}
        
        try:
            # 방 정보
            processed['room_info'] = {
                'width': room_data.get('width', 800),
                'height': room_data.get('height', 600),
                'style': room_data.get('style', 'modern')
            }
            
            # 가구 좌표 변환
            processed['furniture'] = []
            for item in room_data.get('furniture', []):
                mongo_x = item.get('x', 400)
                mongo_y = item.get('y', 300)
                canvas_x, canvas_y = self.mongo_to_canvas(mongo_x, mongo_y)
                
                processed['furniture'].append({
                    'type': item.get('type', 'chair'),
                    'x': canvas_x,
                    'y': canvas_y,
                    'width': item.get('width', 50),
                    'height': item.get('height', 50)
                })
            
            return processed
            
        except Exception as e:
            print(f"방 데이터 처리 오류: {e}")
            return {'room_info': {'width': 800, 'height': 600, 'style': 'modern'}, 'furniture': []}

# 테스트
converter = MongoCoordinateConverter()
test_data = {
    'width': 800,
    'height': 600,
    'style': 'modern',
    'furniture': [
        {'type': 'sofa', 'x': 100, 'y': 500, 'width': 200, 'height': 80},
        {'type': 'table', 'x': 400, 'y': 300, 'width': 100, 'height': 60}
    ]
}

result = converter.process_room_data(test_data)
print("✅ Cell-2 완료: 좌표 변환기 준비")
print(f"변환 결과: {result}")

In [ ]:
# Cell-3: 단순한 ComfyUI 워크플로우 (CPU 최적화)
import uuid
import time

class SimpleComfyUI:
    def __init__(self):
        self.base_workflow = {
            "1": {
                "inputs": {
                    "ckpt_name": "v1-5-pruned-emaonly.ckpt"
                },
                "class_type": "CheckpointLoaderSimple",
                "_meta": {"title": "Load Checkpoint"}
            },
            "2": {
                "inputs": {
                    "text": "modern interior design, living room",
                    "clip": ["1", 1]
                },
                "class_type": "CLIPTextEncode",
                "_meta": {"title": "CLIP Text Encode (Prompt)"}
            },
            "3": {
                "inputs": {
                    "text": "blurry, low quality, bad anatomy",
                    "clip": ["1", 1]
                },
                "class_type": "CLIPTextEncode",
                "_meta": {"title": "CLIP Text Encode (Negative)"}
            },
            "4": {
                "inputs": {
                    "width": 512,
                    "height": 512,
                    "batch_size": 1
                },
                "class_type": "EmptyLatentImage",
                "_meta": {"title": "Empty Latent Image"}
            },
            "5": {
                "inputs": {
                    "seed": 42,
                    "steps": 10,
                    "cfg": 7.0,
                    "sampler_name": "euler",
                    "scheduler": "normal",
                    "denoise": 1.0,
                    "model": ["1", 0],
                    "positive": ["2", 0],
                    "negative": ["3", 0],
                    "latent_image": ["4", 0]
                },
                "class_type": "KSampler",
                "_meta": {"title": "KSampler"}
            },
            "6": {
                "inputs": {
                    "samples": ["5", 0],
                    "vae": ["1", 2]
                },
                "class_type": "VAEDecode",
                "_meta": {"title": "VAE Decode"}
            },
            "7": {
                "inputs": {
                    "filename_prefix": "ComfyUI",
                    "images": ["6", 0]
                },
                "class_type": "SaveImage",
                "_meta": {"title": "Save Image"}
            }
        }
    
    def create_room_prompt(self, room_data):
        """방 데이터를 기반으로 프롬프트 생성"""
        style = room_data.get('room_info', {}).get('style', 'modern')
        furniture_types = [f['type'] for f in room_data.get('furniture', [])]
        
        furniture_text = ', '.join(furniture_types) if furniture_types else 'furniture'
        
        prompt = f"{style} interior design, living room with {furniture_text}, high quality, detailed"
        return prompt
    
    def generate_workflow(self, room_data):
        """방 데이터를 기반으로 워크플로우 생성"""
        workflow = self.base_workflow.copy()
        
        # 프롬프트 업데이트
        prompt = self.create_room_prompt(room_data)
        workflow["2"]["inputs"]["text"] = prompt
        
        # 랜덤 시드
        workflow["5"]["inputs"]["seed"] = int(time.time()) % 10000
        
        return workflow

# 테스트
comfy = SimpleComfyUI()
test_workflow = comfy.generate_workflow(result)
print("✅ Cell-3 완료: ComfyUI 워크플로우 준비")
print(f"생성된 프롬프트: {test_workflow['2']['inputs']['text']}")

In [ ]:
# Cell-4: ComfyUI 서버 실행 (CPU 모드)
import subprocess
import threading
import time
import requests
import os

# 환경변수 설정 (미리보기 비활성화)
os.environ['COLAB_DISABLE_PREVIEW'] = '1'
os.environ['COMFYUI_DISABLE_PREVIEW'] = '1'

class ComfyUIServer:
    def __init__(self):
        self.server_process = None
        self.server_url = "http://127.0.0.1:8188"
        
    def start_server(self):
        """ComfyUI 서버 시작 (CPU 모드)"""
        print("🚀 ComfyUI 서버 시작 중... (CPU 모드)")
        
        cmd = [
            "python", "main.py",
            "--cpu",  # CPU 모드
            "--listen", "127.0.0.1",
            "--port", "8188",
            "--disable-auto-launch",
            "--disable-metadata"
        ]
        
        try:
            self.server_process = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                cwd="/content/ComfyUI"
            )
            
            # 서버 시작 확인
            for i in range(60):  # 60초 대기
                try:
                    response = requests.get(f"{self.server_url}/system_stats", timeout=2)
                    if response.status_code == 200:
                        print("✅ ComfyUI 서버 시작 완료!")
                        return True
                except:
                    pass
                    
                print(f"서버 시작 대기 중... ({i+1}/60)")
                time.sleep(1)
            
            print("❌ 서버 시작 실패")
            return False
            
        except Exception as e:
            print(f"서버 시작 오류: {e}")
            return False
    
    def generate_image(self, workflow):
        """이미지 생성 요청"""
        try:
            # 워크플로우 실행 요청
            response = requests.post(
                f"{self.server_url}/prompt",
                json={"prompt": workflow},
                timeout=300  # 5분 타임아웃
            )
            
            if response.status_code == 200:
                result = response.json()
                prompt_id = result.get('prompt_id')
                print(f"✅ 이미지 생성 시작: {prompt_id}")
                
                # 완료 대기
                for i in range(300):  # 5분 대기
                    try:
                        history_response = requests.get(f"{self.server_url}/history/{prompt_id}")
                        if history_response.status_code == 200:
                            history = history_response.json()
                            if prompt_id in history:
                                print("✅ 이미지 생성 완료!")
                                return True
                    except:
                        pass
                    
                    if i % 10 == 0:
                        print(f"생성 중... ({i+1}/300초)")
                    time.sleep(1)
                
                print("⚠️ 생성 시간 초과")
                return False
            else:
                print(f"❌ 요청 실패: {response.status_code}")
                return False
                
        except Exception as e:
            print(f"이미지 생성 오류: {e}")
            return False

# 서버 시작 및 테스트
server = ComfyUIServer()

# 서버 시작 (백그라운드에서 실행)
def start_server_background():
    server.start_server()

server_thread = threading.Thread(target=start_server_background)
server_thread.daemon = True
server_thread.start()

print("✅ Cell-4 완료: ComfyUI 서버 시작됨")
print("💡 서버가 백그라운드에서 시작됩니다. 약 1-2분 후 사용 가능합니다.")

In [ ]:
# Cell-5: 실제 이미지 생성 테스트
import time

print("🎨 이미지 생성 테스트 시작...")

# 서버 준비 대기
time.sleep(10)

# 테스트용 방 데이터
test_room_data = {
    'room_info': {'width': 800, 'height': 600, 'style': 'modern'},
    'furniture': [
        {'type': 'sofa', 'x': 200, 'y': 100},
        {'type': 'coffee table', 'x': 400, 'y': 300}
    ]
}

# 워크플로우 생성
workflow = comfy.generate_workflow(test_room_data)
print(f"생성할 이미지: {workflow['2']['inputs']['text']}")

# 이미지 생성 실행
success = server.generate_image(workflow)

if success:
    print("🎉 성공! 생성된 이미지를 output 폴더에서 확인하세요.")
    !ls -la output/
else:
    print("❌ 이미지 생성 실패. 서버 로그를 확인해주세요.")

print("\n✅ 모든 테스트 완료!")
print("💡 CPU 모드이므로 생성 시간이 오래 걸릴 수 있습니다.")